# TVDE Lisbon — Driver Analytics
## Notebook 02: SQL Analysis via SQLite

**Author:** Gabriel | Lisbon, Portugal | 2026

---

### Overview

This notebook replicates and extends the Python analysis from notebook 01
using SQL queries via SQLite, demonstrating proficiency in both tools for data analysis.

The same dataset is loaded into an in-memory SQLite database and queried using
standard SQL. Results are consistent with notebook 01 and the Power BI dashboard —
verified by cross-validation.

### Database Setup

| Property | Value |
|---|---|
| Source | tvde_rides_clean.csv (cleaned in notebook 01) |
| Engine | SQLite (in-memory via Python sqlite3 library) |
| Table | rides |
| Columns | 12 |

### SQL Features Used

`CASE WHEN` · `GROUP BY` · `HAVING` · `subqueries` · `UNION ALL`  
`strftime()` · `julianday()` · `ROUND()` · `CAST()`

### Notebook Structure

| Group | Queries | Topic |
|---|---|---|
| Group 1 | Q1.1 – Q1.3 | Overview & Volume |
| Group 2 | Q2.1 – Q2.3 | Earnings & Profitability |
| Group 3 | Q3.1 – Q3.3 | Temporal Patterns |
| Group 4 | Q4.1 – Q4.4 | Acceptance Criteria & Net Profit |

---
## Setup — Load Data & Create Database

Load the cleaned CSV into a pandas DataFrame, apply the same naming
cleanup as notebook 01, then load it into an in-memory SQLite database.

In [1]:
# SQL Analysis — TVDE Lisbon Dataset
# Using SQLite to query the cleaned dataset with SQL

import sqlite3
import pandas as pd

# Load cleaned data
df = pd.read_csv("../data/processed/tvde_rides_clean.csv")
# Fix inconsistent platform naming
df["Platform"] = df["Platform"].str.strip().str.title()

# Fix category naming (same cleanup as main notebook)
df["Category"] = df["Category"].str.strip().str.title()

# Create SQLite database in memory
conn = sqlite3.connect(":memory:")

# Remove auxiliary validation columns if present
cols_to_drop = ["Duration_Calculated", "Duration_Diff"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Load dataframe into SQL table
df.to_sql("rides", conn, index=False, if_exists="replace")

print("Database created successfully.")
print(f"Table 'rides' loaded with {len(df)} rows.")
print(f"\nColumns available:")
for col in df.columns:
    print(f"  - {col}")

Database created successfully.
Table 'rides' loaded with 2541 rows.

Columns available:
  - Ride_ID
  - Platform
  - Category
  - Date
  - Start_Time
  - End_Time
  - Duration_Min
  - Origin_PostCode
  - Dest_PostCode
  - Distance_Km
  - Client_Fare_EUR
  - Driver_Earnings_EUR


---
## Group 1 — Overview & Volume

Three queries to establish a baseline understanding of the dataset:
total volume, platform split and category breakdown.

| Query | Description |
|---|---|
| Q1.1 | General dataset overview (totals and averages) |
| Q1.2 | Rides and earnings broken down by platform |
| Q1.3 | Rides and efficiency broken down by service category |

In [2]:
# ── GROUP 1: OVERVIEW & VOLUME ──────────────────────────────────────

def run_query(query, conn):
    return pd.read_sql_query(query, conn)

# Q1.1 — General dataset overview
q1_1 = """
SELECT
    COUNT(*)                                    AS total_rides,
    COUNT(DISTINCT Date)                        AS days_worked,
    COUNT(DISTINCT Platform)                    AS platforms,
    ROUND(SUM(Distance_Km), 2)                  AS total_km,
    ROUND(SUM(Duration_Min) / 60.0, 2)          AS total_hours,
    ROUND(SUM(Driver_Earnings_EUR), 2)          AS total_gross_earnings,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings_per_ride,
    ROUND(MIN(Driver_Earnings_EUR), 2)          AS min_ride_earnings,
    ROUND(MAX(Driver_Earnings_EUR), 2)          AS max_ride_earnings
FROM rides;
"""
print("Q1.1 — Dataset Overview")
print("-" * 55)
print(run_query(q1_1, conn).to_string(index=False))

# Q1.2 — Rides and earnings by platform
q1_2 = """
SELECT
    Platform,
    COUNT(*)                                    AS total_rides,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rides), 1)
                                                AS pct_rides,
    ROUND(SUM(Driver_Earnings_EUR), 2)          AS total_earnings,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings_per_ride,
    ROUND(SUM(Distance_Km), 2)                  AS total_km,
    ROUND(AVG(Distance_Km), 2)                  AS avg_km_per_ride
FROM rides
GROUP BY Platform
ORDER BY total_rides DESC;
"""
print("\nQ1.2 — Rides and Earnings by Platform")
print("-" * 55)
print(run_query(q1_2, conn).to_string(index=False))

# Q1.3 — Rides by category
q1_3 = """
SELECT
    Category,
    COUNT(*)                                    AS total_rides,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rides), 1)
                                                AS pct_rides,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings,
    ROUND(AVG(Distance_Km), 2)                  AS avg_km,
    ROUND(AVG(Driver_Earnings_EUR / Distance_Km), 2)
                                                AS avg_eur_per_km
FROM rides
GROUP BY Category
ORDER BY total_rides DESC;
"""
print("\nQ1.3 — Rides by Category")
print("-" * 55)
print(run_query(q1_3, conn).to_string(index=False))

Q1.1 — Dataset Overview
-------------------------------------------------------
 total_rides  days_worked  platforms  total_km  total_hours  total_gross_earnings  avg_earnings_per_ride  min_ride_earnings  max_ride_earnings
        2541          152          2  23951.11       679.75               15986.7                   6.29               2.89              23.75

Q1.2 — Rides and Earnings by Platform
-------------------------------------------------------
Platform  total_rides  pct_rides  total_earnings  avg_earnings_per_ride  total_km  avg_km_per_ride
    Bolt         1272       50.1         7918.26                   6.23  11507.59             9.05
    Uber         1269       49.9         8068.44                   6.36  12443.52             9.81

Q1.3 — Rides by Category
-------------------------------------------------------
       Category  total_rides  pct_rides  avg_earnings  avg_km  avg_eur_per_km
         Uber X          985       38.8          6.38   10.40            0.77
    

### Group 1 — Key Observations

- **Q1.1** confirms the total dataset volume, km and gross earnings for cross-validation
- **Q1.2** shows the platform split — Uber and Bolt are almost perfectly balanced in ride count (49.9% vs 50.1%)
- **Q1.3** reveals that Uber X dominates in volume (~39%) but Uber X Priority
  leads in €/km efficiency among the main categories

---
## Group 2 — Earnings & Profitability

Three queries focused on when and how earnings are generated.

| Query | Description |
|---|---|
| Q2.1 | Top 10 highest-earning days |
| Q2.2 | Daily performance categorised vs the 100€ gross target |
| Q2.3 | Earnings efficiency (€/km and €/hour) by distance bracket |

> **Note on Q2.2:** SQLite does not support nested aggregates directly.
> A subquery is used to first calculate daily totals, then categorise them.

In [3]:
# ── GROUP 2: EARNINGS & PROFITABILITY ───────────────────────────────

# Q2.1 — Top 10 best earning days
q2_1 = """
SELECT
    Date,
    COUNT(*)                                    AS total_rides,
    ROUND(SUM(Driver_Earnings_EUR), 2)          AS daily_gross,
    ROUND(SUM(Distance_Km), 2)                  AS daily_km,
    ROUND(SUM(Duration_Min) / 60.0, 2)          AS daily_hours,
    ROUND(SUM(Driver_Earnings_EUR) /
          (SUM(Duration_Min) / 60.0), 2)        AS eur_per_hour
FROM rides
GROUP BY Date
ORDER BY daily_gross DESC
LIMIT 10;
"""
print("Q2.1 — Top 10 Best Earning Days")
print("-" * 65)
print(run_query(q2_1, conn).to_string(index=False))

# Q2.2 — Daily earnings above and below target
q2_2 = """
SELECT
    CASE
        WHEN SUM(Driver_Earnings_EUR) >= 100 THEN 'Above target (≥100€)'
        WHEN SUM(Driver_Earnings_EUR) >= 80  THEN 'Close (80–100€)'
        ELSE 'Below (< 80€)'
    END                                         AS performance,
    COUNT(*)                                    AS days,
    ROUND(AVG(SUM(Driver_Earnings_EUR)), 2)     AS avg_earnings
FROM rides
GROUP BY Date
GROUP BY performance
ORDER BY avg_earnings DESC;
"""

# Simplified version for SQLite compatibility
q2_2 = """
SELECT
    performance,
    COUNT(*)                                    AS days,
    ROUND(AVG(daily_gross), 2)                  AS avg_earnings
FROM (
    SELECT
        Date,
        SUM(Driver_Earnings_EUR)                AS daily_gross,
        CASE
            WHEN SUM(Driver_Earnings_EUR) >= 100
                THEN 'Above target (100+ EUR)'
            WHEN SUM(Driver_Earnings_EUR) >= 80
                THEN 'Close (80-100 EUR)'
            ELSE 'Below (under 80 EUR)'
        END                                     AS performance
    FROM rides
    GROUP BY Date
)
GROUP BY performance
ORDER BY avg_earnings DESC;
"""
print("\nQ2.2 — Daily Performance vs Target")
print("-" * 65)
print(run_query(q2_2, conn).to_string(index=False))

# Q2.3 — Earnings efficiency by distance bracket
q2_3 = """
SELECT
    CASE
        WHEN Distance_Km < 3    THEN '0-3 km'
        WHEN Distance_Km < 5    THEN '3-5 km'
        WHEN Distance_Km < 8    THEN '5-8 km'
        WHEN Distance_Km < 12   THEN '8-12 km'
        WHEN Distance_Km < 18   THEN '12-18 km'
        WHEN Distance_Km < 25   THEN '18-25 km'
        ELSE '25+ km'
    END                                         AS distance_bracket,
    COUNT(*)                                    AS total_rides,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings,
    ROUND(AVG(Driver_Earnings_EUR /
              Distance_Km), 2)                  AS avg_eur_per_km,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)            AS avg_eur_per_hour
FROM rides
GROUP BY distance_bracket
ORDER BY
    CASE distance_bracket
        WHEN '0-3 km'   THEN 1
        WHEN '3-5 km'   THEN 2
        WHEN '5-8 km'   THEN 3
        WHEN '8-12 km'  THEN 4
        WHEN '12-18 km' THEN 5
        WHEN '18-25 km' THEN 6
        ELSE 7
    END;
"""
print("\nQ2.3 — Earnings Efficiency by Distance Bracket")
print("-" * 65)
print(run_query(q2_3, conn).to_string(index=False))

Q2.1 — Top 10 Best Earning Days
-----------------------------------------------------------------
      Date  total_rides  daily_gross  daily_km  daily_hours  eur_per_hour
2026-07-03           22       159.84    151.35         5.35         29.88
2026-06-03           19       155.80    182.42         5.85         26.63
2026-05-24           23       153.20    245.85         6.13         24.98
2026-07-06           22       148.23    145.65         4.55         32.58
2026-07-02           21       145.51    143.84         4.55         31.98
2026-05-29           20       144.93    151.45         4.45         32.57
2026-05-30           19       133.04    181.69         4.33         30.70
2026-07-19           19       132.77    226.81         5.00         26.55
2026-07-04           22       132.41    172.66         4.92         26.93
2026-03-08           22       130.69    187.41         5.00         26.14

Q2.2 — Daily Performance vs Target
----------------------------------------------------

### Group 2 — Key Observations

- **Q2.1** shows the top earning days and their €/hour — the best days are not
  always the longest, confirming that ride quality matters more than hours worked
- **Q2.2** confirms that the majority of days exceed the 100€ gross target,
  with average earnings well above the threshold on good days
- **Q2.3** shows short rides (0–3 km) yield the highest €/km and €/hour of any
  distance bracket

---
## Group 3 — Temporal Patterns

Three queries analysing how earnings and acceptance rates vary across
days of the week, hours of the day and months.

| Query | Description |
|---|---|
| Q3.1 | Average daily earnings and €/hour by day of week |
| Q3.2 | Performance and acceptance rate by hour of day |
| Q3.3 | Monthly progression of total earnings and €/hour |

> **Note on Q3.1:** A JOIN between a daily-aggregated subquery and the rides
> table is used to correctly attribute day-of-week labels to each day's totals.

In [4]:
# ── GROUP 3: TEMPORAL PATTERNS ──────────────────────────────────────

# Q3.1 — Average earnings by day of week
q3_1 = """
SELECT
    CASE CAST(strftime('%w', Date) AS INTEGER)
        WHEN 0 THEN '7-Sunday'
        WHEN 1 THEN '1-Monday'
        WHEN 2 THEN '2-Tuesday'
        WHEN 3 THEN '3-Wednesday'
        WHEN 4 THEN '4-Thursday'
        WHEN 5 THEN '5-Friday'
        WHEN 6 THEN '6-Saturday'
    END                                         AS day_of_week,
    COUNT(DISTINCT Date)                        AS days_worked,
    COUNT(*)                                    AS total_rides,
    ROUND(AVG(daily_gross), 2)                  AS avg_daily_earnings,
    ROUND(AVG(daily_eur_per_hour), 2)           AS avg_eur_per_hour
FROM (
    SELECT
        Date,
        SUM(Driver_Earnings_EUR)                AS daily_gross,
        SUM(Driver_Earnings_EUR) /
            (SUM(Duration_Min) / 60.0)          AS daily_eur_per_hour
    FROM rides
    GROUP BY Date
) daily
JOIN rides r USING (Date)
GROUP BY day_of_week
ORDER BY day_of_week;
"""
print("Q3.1 — Average Earnings by Day of Week")
print("-" * 65)
print(run_query(q3_1, conn).to_string(index=False))

# Q3.2 — Best hours of the day
q3_2 = """
SELECT
    CAST(strftime('%H', Start_Time) AS INTEGER) AS hour,
    COUNT(*)                                    AS total_rides,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings,
    ROUND(AVG(Driver_Earnings_EUR /
              Distance_Km), 2)                  AS avg_eur_per_km,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)            AS avg_eur_per_hour,
    ROUND(SUM(CASE WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
                   AND Driver_Earnings_EUR / Duration_Min * 60 >= 12
                   THEN 1 ELSE 0 END) * 100.0 /
              COUNT(*), 1)                      AS acceptance_rate_pct
FROM rides
WHERE CAST(strftime('%H', Start_Time) AS INTEGER) >= 9
GROUP BY hour
HAVING total_rides >= 10
ORDER BY hour;
"""
print("\nQ3.2 — Performance by Hour of Day")
print("-" * 65)
print(run_query(q3_2, conn).to_string(index=False))

# Q3.3 — Monthly progression
q3_3 = """
SELECT
    strftime('%Y-%m', Date)                     AS month,
    COUNT(DISTINCT Date)                        AS days_worked,
    COUNT(*)                                    AS total_rides,
    ROUND(SUM(Driver_Earnings_EUR), 2)          AS total_earnings,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)            AS avg_eur_per_hour,
    ROUND(SUM(Driver_Earnings_EUR) /
              COUNT(DISTINCT Date), 2)          AS avg_daily_earnings
FROM rides
GROUP BY month
ORDER BY month;
"""
print("\nQ3.3 — Monthly Progression")
print("-" * 65)
print(run_query(q3_3, conn).to_string(index=False))


Q3.1 — Average Earnings by Day of Week
-----------------------------------------------------------------
day_of_week  days_worked  total_rides  avg_daily_earnings  avg_eur_per_hour
   1-Monday           23          402              108.72             23.19
  2-Tuesday           21          351              104.46             23.23
3-Wednesday           12          199              112.82             22.80
 4-Thursday           22          361              100.17             23.09
   5-Friday           25          426              116.73             24.59
 6-Saturday           24          399              107.48             24.35
   7-Sunday           25          403              107.59             24.80

Q3.2 — Performance by Hour of Day
-----------------------------------------------------------------
 hour  total_rides  avg_earnings  avg_eur_per_km  avg_eur_per_hour  acceptance_rate_pct
    9           13          7.21            0.88             27.78                 84.6
   11     

### Group 3 — Key Observations

- **Q3.1** confirms Friday leads on total daily earnings, with Sunday close
  behind on €/hour (24.80 vs 24.59)
- **Q3.2** identifies 16:00–18:00 as the golden window — highest acceptance rate
  (91–94%) combined with strong ride volume
- **Q3.3** shows €/hour improved from March through July, before dipping in
  August (25.92 €/h, down from July's 28.18 €/h) — a positive but not
  perfectly linear trend

---
## Group 4 — Acceptance Criteria & Net Profit

Four queries testing the ride acceptance strategy and calculating net profit
after all operational costs.

| Query | Description |
|---|---|
| Q4.1 | Breakdown of rides by acceptance criteria status |
| Q4.2 | Acceptance rate and €/hour by platform |
| Q4.3 | Top 10 most efficient individual rides (by €/hour) |
| Q4.4 | Net profit after fuel and fleet costs |

> **Acceptance criteria:** Driver_Earnings ≥ 0.50 €/km AND ≥ 12 €/hour (both required)

> **Net profit formula:**
> ```
> Gross Earnings − (Total_Km × 0.0976 €/km) − (ROUND(days_range / 7) × 30 €)
> ```
> Where `julianday()` is used to calculate the date range in days.

In [5]:
# ── GROUP 4: ACCEPTANCE CRITERIA & PROFITABILITY ────────────────────

# Q4.1 — Acceptance criteria compliance overall
q4_1 = """
SELECT
    criteria_status,
    COUNT(*)                                    AS total_rides,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rides), 1)
                                                AS pct_rides,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings,
    ROUND(AVG(Driver_Earnings_EUR /
              Distance_Km), 2)                  AS avg_eur_per_km,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)            AS avg_eur_per_hour
FROM (
    SELECT *,
        CASE
            WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
             AND Driver_Earnings_EUR / Duration_Min * 60 >= 12
                THEN 'Meets both criteria'
            WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
                THEN 'Only EUR/km met'
            WHEN Driver_Earnings_EUR / Duration_Min * 60 >= 12
                THEN 'Only EUR/hour met'
            ELSE 'Neither met'
        END AS criteria_status
    FROM rides
)
GROUP BY criteria_status
ORDER BY total_rides DESC;
"""
print("Q4.1 — Acceptance Criteria Compliance")
print("-" * 75)
print(run_query(q4_1, conn).to_string(index=False))

# Q4.2 — Acceptance rate by platform
q4_2 = """
SELECT
    Platform,
    COUNT(*)                                    AS total_rides,
    SUM(CASE WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
              AND Driver_Earnings_EUR / Duration_Min * 60 >= 12
             THEN 1 ELSE 0 END)                 AS meets_both,
    ROUND(SUM(CASE WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
                    AND Driver_Earnings_EUR / Duration_Min * 60 >= 12
                   THEN 1 ELSE 0 END) * 100.0 /
              COUNT(*), 1)                      AS acceptance_rate_pct,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)            AS avg_eur_per_hour
FROM rides
GROUP BY Platform
ORDER BY acceptance_rate_pct DESC;
"""
print("\nQ4.2 — Acceptance Rate by Platform")
print("-" * 75)
print(run_query(q4_2, conn).to_string(index=False))

# Q4.3 — Top 10 most profitable individual rides
q4_3 = """
SELECT
    Ride_ID,
    Platform,
    Category,
    Date,
    ROUND(Distance_Km, 2)                       AS km,
    ROUND(Duration_Min, 0)                      AS duration_min,
    ROUND(Driver_Earnings_EUR, 2)               AS earnings,
    ROUND(Driver_Earnings_EUR / Distance_Km, 2) AS eur_per_km,
    ROUND(Driver_Earnings_EUR /
          Duration_Min * 60, 2)                 AS eur_per_hour
FROM rides
ORDER BY eur_per_hour DESC
LIMIT 10;
"""
print("\nQ4.3 — Top 10 Most Efficient Rides (by €/hour)")
print("-" * 75)
print(run_query(q4_3, conn).to_string(index=False))

# Q4.4 — Net profit summary with fuel cost
q4_4 = """
SELECT
    COUNT(DISTINCT Date)                        AS days_worked,
    ROUND(SUM(Driver_Earnings_EUR), 2)          AS total_gross,
    ROUND(SUM(Distance_Km) * 0.0976, 2)        AS total_fuel_cost,
    CAST(ROUND((julianday(MAX(Date)) -
                julianday(MIN(Date))) / 7)
         AS INTEGER) * 30                       AS total_fleet_cost,
    ROUND(SUM(Driver_Earnings_EUR) -
          SUM(Distance_Km) * 0.0976, 2)        AS net_before_fleet,
    ROUND(SUM(Driver_Earnings_EUR) -
          SUM(Distance_Km) * 0.0976 -
          (CAST(ROUND((julianday(MAX(Date)) -
                       julianday(MIN(Date))) / 7)
           AS INTEGER) * 30), 2)               AS net_after_all_costs,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)           AS avg_eur_per_hour
FROM rides;
"""
print("\nQ4.4 — Net Profit Summary")
print("-" * 75)
print(run_query(q4_4, conn).to_string(index=False))

Q4.1 — Acceptance Criteria Compliance
---------------------------------------------------------------------------
    criteria_status  total_rides  pct_rides  avg_earnings  avg_eur_per_km  avg_eur_per_hour
Meets both criteria         2143       84.3          6.02            0.94             26.52
  Only EUR/hour met          392       15.4          7.77            0.44             22.02
    Only EUR/km met            5        0.2          6.13            0.63             10.45
        Neither met            1        0.0          4.69            0.48             10.82

Q4.2 — Acceptance Rate by Platform
---------------------------------------------------------------------------
Platform  total_rides  meets_both  acceptance_rate_pct  avg_eur_per_hour
    Bolt         1272        1122                 88.2             25.27
    Uber         1269        1021                 80.5             26.30

Q4.3 — Top 10 Most Efficient Rides (by €/hour)
-----------------------------------------------

### Group 4 — Key Observations

- **Q4.1** confirms 84.3% of rides meet both acceptance criteria — the strategy is working
- **Q4.2** shows Bolt with a clear lead over Uber in acceptance rate (88.2% vs 80.5%,
  a ~7.7-point gap)
- **Q4.3** reveals that the most efficient rides by €/hour are all very short
  (under 2.3 km, 2–4 minutes) — confirming the 0–3 km bracket finding
- **Q4.4** provides the net profit figure used for cross-validation across all three tools

---
## Final Summary Query

A single consolidated query returning the most important metrics from the full dataset.
Used as the cross-validation reference point against Power BI and notebook 01.

In [6]:
# ── FINAL SUMMARY ───────────────────────────────────────────────────

summary = """
SELECT
    'Total Rides'           AS metric,
    CAST(COUNT(*) AS TEXT)  AS value
FROM rides
UNION ALL
SELECT 'Total Gross (EUR)',
    CAST(ROUND(SUM(Driver_Earnings_EUR), 2) AS TEXT)
FROM rides
UNION ALL
SELECT 'Net Profit after All Costs (EUR)',
    CAST(ROUND(SUM(Driver_Earnings_EUR) -
         SUM(Distance_Km) * 0.0976 -
         (CAST(ROUND((julianday(MAX(Date)) -
                      julianday(MIN(Date))) / 7)
          AS INTEGER) * 30), 2) AS TEXT)
FROM rides
UNION ALL
SELECT 'Avg EUR/hour',
    CAST(ROUND(AVG(Driver_Earnings_EUR /
         Duration_Min * 60), 2) AS TEXT)
FROM rides
UNION ALL
SELECT 'Best Platform (acceptance rate)',
    Platform
FROM (
    SELECT Platform,
           SUM(CASE WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
                     AND Driver_Earnings_EUR / Duration_Min * 60 >= 12
                    THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS rate
    FROM rides GROUP BY Platform ORDER BY rate DESC LIMIT 1
)
UNION ALL
SELECT 'Best Day of Week',
    day_of_week
FROM (
    SELECT
        CASE CAST(strftime('%w', Date) AS INTEGER)
            WHEN 0 THEN 'Sunday'
            WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'
            WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'
            WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
        END AS day_of_week,
        AVG(daily_gross) AS avg_gross
    FROM (
        SELECT Date,
               SUM(Driver_Earnings_EUR) AS daily_gross
        FROM rides GROUP BY Date
    ) d
    JOIN rides r USING(Date)
    GROUP BY day_of_week
    ORDER BY avg_gross DESC
    LIMIT 1
)
UNION ALL
SELECT 'Golden Window', '16:00 - 18:00 (91-93% acceptance rate)';
"""

print("=" * 75)
print("SQL ANALYSIS — FINAL SUMMARY")
print("=" * 75)
print(run_query(summary, conn).to_string(index=False))
print("=" * 75)

SQL ANALYSIS — FINAL SUMMARY
                          metric                                  value
                     Total Rides                                   2541
               Total Gross (EUR)                                15986.7
Net Profit after All Costs (EUR)                               12869.07
                    Avg EUR/hour                                  25.78
 Best Platform (acceptance rate)                                   Bolt
                Best Day of Week                                 Friday
                   Golden Window 16:00 - 18:00 (91-93% acceptance rate)


### Key SQL Findings

*(Full March–August 2026 dataset)*

| Metric | Value |
|---|---|
| Total rides | 2,541 |
| Total gross earnings | 15,986.70 € |
| Total fuel cost | 2,337.63 € |
| Total fleet cost | 780.00 € |
| Net profit after all costs | 12,869.07 € |
| Average €/hour | 25.78 |
| Best platform (acceptance rate) | Bolt (88.2%) |
| Best day of week | Friday |
| Golden window | 16:00–18:00 (91–94% acceptance rate) |

### Cross-Validation

Net profit after all costs is validated against Power BI and notebook 01.
All three tools use identical formulas — any mismatch indicates a formula discrepancy.

> Compare the `net_after_all_costs` value from Q4.4 with:
> - Power BI → `Net_Profit_EUR` measure on Page 1
> - Python → Final Validation cell in notebook 01